[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ai-agents-certified/notebooks/day-04-tool-calling-react.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · ReAct Agents and Tool Calling with ToolNode
**certified-journeys / ai-agents-certified** · Day 4 · Tool Calling

> **Goal for today:** Build a ReAct agent manually using ToolNode, wire three real tools (calculator, DuckDuckGo search, Python REPL), and implement a max-iterations safety guard so the agent never loops forever.


In [ ]:
%pip install -q langgraph langchain-openai langchain-community duckduckgo-search numexpr


## Step 1 · The ReAct Pattern — Reason, Act, Observe

ReAct (Reason + Act) is the dominant agentic loop pattern:

| Phase | What happens | Graph node |
|-------|-------------|------------|
| **Reason** | LLM decides which tool to call and with what args | `agent` node |
| **Act** | Tool is executed, result returned as `ToolMessage` | `tools` node (ToolNode) |
| **Observe** | LLM receives the tool result and decides next step | back to `agent` |

LangGraph's `ToolNode` handles the **Act** phase: it looks at the most recent `AIMessage`, finds any `tool_calls`, executes each tool, and appends `ToolMessage` results to state — all in one node.

`create_react_agent` is a one-liner shortcut, but building it manually reveals the wiring. **We build manually today.**


In [ ]:
import os
from typing import Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing_extensions import TypedDict

# We'll use gpt-4o-mini — cheap, fast, reliable tool-calling support.
# Swap to any OpenAI-compatible endpoint by changing the base_url.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("LLM loaded:", llm.model_name)


**What just happened?**
- `ChatOpenAI` wraps the OpenAI chat completions endpoint.
- `temperature=0` makes the model deterministic — essential for reliable tool selection.
- `ToolNode` (imported but not wired yet) will be the Act node in our graph.


## Step 2 · Define Three Tools

LangGraph tools are plain Python functions decorated with `@tool`. The docstring becomes the tool description the LLM reads to decide when to use it — **write clear, specific docstrings**.

| Tool | Purpose | Library |
|------|---------|--------|
| `calculator` | Safe arithmetic evaluation | `numexpr` |
| `web_search` | Live web results | `duckduckgo-search` |
| `python_repl` | Arbitrary Python execution | built-in `exec` |

> **Production note:** `python_repl` with `exec` is dangerous in untrusted environments. In production, use a sandboxed subprocess or `RestrictedPython`.


In [ ]:
import numexpr
from langchain_core.tools import tool
from duckduckgo_search import DDGS

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely. Use for arithmetic, percentages, and unit conversions.
    Input must be a valid Python math expression (e.g. '(3 + 4) * 2 / 100').
    Returns the numeric result as a string.
    """
    try:
        result = numexpr.evaluate(expression).item()  # .item() converts numpy scalar → Python float
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"


@tool
def web_search(query: str) -> str:
    """Search the web for current information using DuckDuckGo.
    Use for factual lookups, current events, or anything that requires up-to-date knowledge.
    Returns up to 3 snippet results.
    """
    try:
        results = DDGS().text(query, max_results=3)
        if not results:
            return "No results found."
        # Format each result as 'Title: ... Body: ...'
        return "\n\n".join(
            f"Title: {r['title']}\nBody: {r['body']}" for r in results
        )
    except Exception as e:
        return f"Search error: {e}"


@tool
def python_repl(code: str) -> str:
    """Execute a snippet of Python code and return printed output.
    Use for data manipulation, string formatting, or logic that can't be expressed as a math expression.
    Always use print() to produce output — return values are not captured.
    WARNING: Runs in the same process — do not use in untrusted environments.
    """
    import io, sys
    captured = io.StringIO()
    sys.stdout = captured
    try:
        exec(code, {})  # isolated namespace dict prevents polluting globals
    except Exception as e:
        sys.stdout = sys.__stdout__
        return f"Error: {e}"
    finally:
        sys.stdout = sys.__stdout__
    return captured.getvalue() or "(no output)"


tools = [calculator, web_search, python_repl]

# Smoke-test the calculator
print(calculator.invoke({"expression": "(100 + 200) * 0.15"}))


**What just happened?**
- `@tool` introspects the function signature and docstring to generate a JSON schema the LLM receives.
- `numexpr.evaluate` is safer than Python's `eval` because it only supports mathematical expressions.
- **`python_repl` captures stdout** via `io.StringIO` — this is how you safely surface `print()` output.
- The smoke-test `(100 + 200) * 0.15` should print `45.0`.


## Step 3 · Define State and Build the Graph

Our state needs two things:
1. `messages` — the conversation history (with `add_messages` reducer so new messages are appended, not overwritten).
2. `iteration_count` — a counter we increment each time the agent node runs, used by the max-iterations guard.

Graph edges:
```
START → agent → (tool_calls?) → tools → agent → ... → END
                     ↓ no
                    END
```


In [ ]:
from langgraph.graph import START

MAX_ITERATIONS = 6  # safety ceiling — agent stops after this many tool calls

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]  # add_messages reducer: append, never replace
    iteration_count: int


# Bind tools to the LLM so it knows their schemas
llm_with_tools = llm.bind_tools(tools)


def agent_node(state: AgentState) -> dict:
    """Reason step: LLM decides whether to call a tool or produce a final answer."""
    response = llm_with_tools.invoke(state["messages"])
    return {
        "messages": [response],
        "iteration_count": state["iteration_count"] + 1,
    }


def should_continue(state: AgentState) -> str:
    """Routing function: decides whether to call tools, stop (answer ready), or stop (max iterations hit)."""
    last = state["messages"][-1]  # the most recent LLM response

    # Safety valve: if we've hit the ceiling, force-stop regardless of tool calls
    if state["iteration_count"] >= MAX_ITERATIONS:
        print(f"[guard] Max iterations ({MAX_ITERATIONS}) reached — stopping.")
        return "end"

    # If the LLM produced tool calls, route to the ToolNode
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"

    # Otherwise the LLM produced a final answer
    return "end"


# ToolNode automatically executes every tool_call on the last AIMessage
tool_node = ToolNode(tools)

# Build the graph
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", "end": END},  # map routing strings to actual nodes
)
builder.add_edge("tools", "agent")  # after tool execution, always reason again

graph = builder.compile()
print("Graph compiled successfully")


**What just happened?**
- `bind_tools` attaches the tool schemas to the LLM request — without this, the LLM can't call tools.
- `should_continue` is a **conditional edge function**: it inspects state and returns a string key that maps to the next node.
- **`MAX_ITERATIONS` check runs before the tool-call check** — the ceiling wins regardless of what the LLM wants.
- `ToolNode` handles the entire Act phase: deserializes the tool call args, invokes the function, wraps the result in a `ToolMessage`.


## Step 4 · Force the Agent to Use the Calculator on a Multi-Step Word Problem

We craft a word problem that **requires multiple arithmetic steps**. The agent must chain tool calls, and we log the full tool-call chain as it executes.


In [ ]:
WORD_PROBLEM = """
A store sells three items:
  - Widget A: $45.99 each, 12 units sold
  - Widget B: $129.50 each, 7 units sold
  - Widget C: $8.75 each, 83 units sold

After totalling revenue from all three items, apply a 7.5% sales tax.
Then subtract a $50 promotional discount.
What is the final amount the customer owes?

Use the calculator tool for every arithmetic step. Show your work.
"""

initial_state = {
    "messages": [HumanMessage(content=WORD_PROBLEM)],
    "iteration_count": 0,
}

# stream_mode="values" gives us the full state after every node execution
print("=" * 60)
print("RUNNING REACT AGENT")
print("=" * 60)

tool_call_log = []  # we'll build this as we stream

for step_state in graph.stream(initial_state, stream_mode="values"):
    last_msg = step_state["messages"][-1]
    itr = step_state["iteration_count"]

    if isinstance(last_msg, AIMessage):
        if last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                print(f"\n[iter {itr}] TOOL CALL → {tc['name']}({tc['args']})")
                tool_call_log.append({"iteration": itr, "tool": tc["name"], "args": tc["args"]})
        else:
            print(f"\n[iter {itr}] FINAL ANSWER:")
            print(last_msg.content)

    elif isinstance(last_msg, ToolMessage):
        print(f"         ↳ RESULT: {last_msg.content}")

print("\n" + "=" * 60)
print(f"Total tool calls logged: {len(tool_call_log)}")
for entry in tool_call_log:
    print(f"  iter={entry['iteration']} | {entry['tool']} | args={entry['args']}")


**What just happened?**
- `graph.stream` yields state after every node execution — ideal for observability.
- Each `AIMessage` with `tool_calls` is the **Reason** step; each `ToolMessage` is the **Observe** step.
- The tool-call log gives you a machine-readable audit trail of every action the agent took.
- **Notice:** the agent chains multiple calculator calls to break the problem into sub-steps — this is the ReAct pattern in action.


## Step 5 · Demonstrate the Max-Iterations Guard

We temporarily lower `MAX_ITERATIONS` and give the agent a task that would normally require many steps, verifying it stops gracefully instead of looping.


In [ ]:
# Override MAX_ITERATIONS for this test — 2 is low enough to trigger the guard quickly
MAX_ITERATIONS = 2

# Rebuild graph so the closure inside should_continue picks up the new value
builder2 = StateGraph(AgentState)
builder2.add_node("agent", agent_node)
builder2.add_node("tools", tool_node)
builder2.add_edge(START, "agent")
builder2.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
builder2.add_edge("tools", "agent")
graph2 = builder2.compile()

long_task = HumanMessage(
    content="Search the web for the top 5 programming languages in 2024, "
            "then calculate the sum of their Stack Overflow survey percentages, "
            "then write Python code that prints them sorted by popularity."
)

result = graph2.invoke(
    {"messages": [long_task], "iteration_count": 0}
)

print("Agent stopped after", result["iteration_count"], "iterations")
print("Last message type:", type(result["messages"][-1]).__name__)

# Restore to normal
MAX_ITERATIONS = 6


**What just happened?**
- With `MAX_ITERATIONS = 2`, the agent hits the ceiling before completing all three sub-tasks.
- The `[guard]` log line fires and `should_continue` returns `"end"`, routing to `END` without calling any more tools.
- **The agent exits cleanly** — no exception, no infinite loop, just a truncated answer.
- In production, you'd also log a warning and potentially surface a partial result to the user.


## Step 6 · Inspect the Tool Call Schema the LLM Sees

Understanding what the LLM receives helps you write better tool docstrings and debug unexpected tool selection.


In [ ]:
import json

# ChatOpenAI exposes the tool schemas via kwargs inspection after bind_tools
# We can also get them directly from the tool objects
print("Tool schemas sent to the LLM:\n")
for t in tools:
    schema = {
        "name": t.name,
        "description": t.description,
        "parameters": t.args_schema.schema() if t.args_schema else {},
    }
    print(json.dumps(schema, indent=2))
    print()


**What just happened?**
- Each `@tool` function generates a JSON Schema from its type hints and docstring.
- **The description field is the LLM's only guidance** for when to pick this tool — vague descriptions cause wrong tool selections.
- `args_schema` is a Pydantic model; `.schema()` returns the JSON Schema dict.
- You can tune tool selection by tightening the description (e.g., "Use ONLY for arithmetic, not for string operations").


## Step 7 · Run All Three Tools in a Single Query

We give the agent a composite task that naturally requires all three tools to complete, then verify each tool appears in the trace.


In [ ]:
MAX_ITERATIONS = 10  # generous ceiling for this multi-tool test

# Rebuild with updated ceiling
builder3 = StateGraph(AgentState)
builder3.add_node("agent", agent_node)
builder3.add_node("tools", tool_node)
builder3.add_edge(START, "agent")
builder3.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
builder3.add_edge("tools", "agent")
graph3 = builder3.compile()

multi_tool_query = HumanMessage(
    content="""
    Do all three of these:
    1. Use the calculator to compute 17 * 23 + 456 / 12.
    2. Use the web search to find the current population of Tokyo.
    3. Use the Python REPL to print the first 10 Fibonacci numbers.
    Summarize all three results at the end.
    """
)

result3 = graph3.invoke(
    {"messages": [multi_tool_query], "iteration_count": 0}
)

# Count which tools were used
tools_used = set()
for msg in result3["messages"]:
    if isinstance(msg, AIMessage) and msg.tool_calls:
        for tc in msg.tool_calls:
            tools_used.add(tc["name"])

print("Tools used in this run:", tools_used)
print("\nFinal answer:")
print(result3["messages"][-1].content)


**What just happened?**
- The agent autonomously sequenced three different tools to complete three sub-tasks.
- **ToolNode handles multiple tool calls per iteration** — if the LLM emits several `tool_calls` in one `AIMessage`, ToolNode runs them all before returning.
- Iterating `result['messages']` and checking `isinstance(msg, AIMessage)` is the standard way to audit the tool-call chain post-run.


In [ ]:
# Challenge: Build a ReAct agent that uses the calculator to solve this problem
# without any additional guidance beyond the tools you've already defined.
#
# Problem: A train travels at 120 km/h for 2.5 hours, then slows to 80 km/h
# for another 1.75 hours. What is the total distance? What is the average speed
# over the entire journey? Use only the calculator tool.
#
# Requirements:
#   1. Use the graph3 you built above (or rebuild with MAX_ITERATIONS=6)
#   2. After the run, print each calculator call and its result
#   3. Verify the final answer matches manual calculation

# Your solution here:
challenge_question = HumanMessage(
    content=(
        # TODO: paste the problem statement here and invoke graph3
        "A train travels at 120 km/h for 2.5 hours, then slows to 80 km/h "
        "for 1.75 hours. Compute total distance and average speed using the "
        "calculator tool only."
    )
)

# TODO: invoke graph3 with challenge_question
# TODO: loop through messages and print calculator calls + results
# TODO: print the final answer


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| ReAct loop | Reason → Act → Observe, repeated until no tool calls remain |
| `ToolNode` | Handles the entire Act phase: deserializes args, runs tool, wraps result in `ToolMessage` |
| `bind_tools` | Attaches JSON schemas to LLM requests — required for tool calling |
| `add_messages` reducer | Appends messages to state instead of replacing — critical for chat history |
| Max-iterations guard | Check iteration count **before** tool-call check in `should_continue` |
| Tool docstrings | The LLM's only signal for tool selection — write them like API docs |
| `stream_mode="values"` | Yields full state after every node — best for observability |

> **Tip:** Always add a `max_iterations` ceiling in the tool-call loop. LLMs occasionally get stuck repeating the same tool call — a ceiling is the safety valve that makes agents production-safe.

---
## What's next
**Day 5** → Human-in-the-Loop: interrupting the graph before dangerous tool calls, waiting for user approval, and resuming from a saved snapshot with the same thread_id.

Mark Day 4 complete in your [tracker](../index.html).
